# Final Project Testbed

## Challenge 1

Train on 50 images per class from two randomly selected CIFAR-10 classes. This testbed uses 350 separate validation images per class. These images may be used for evaluation and hyperparameter selection, but not to train the model. Final evaluation splits will be provided separately.

You may modify this testbed within the constraints in the project instructions. The example is a small Vision Transformer trained from scratch; other approaches are allowed.


In [ ]:
%pip install -q timm==1.0.30


### Training and evaluation

The same training and evaluation functions are used for both challenges.


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import timm
from timm.models.vision_transformer import VisionTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = list(range(10))  # Use one seed while debugging; report multiple instances.
SCRATCH_EPOCHS = 100
TRANSFER_EPOCHS = 10
BATCH_SIZE = 32

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train(model, loader, optimizer, frozen_backbone=False):
    # Keep a frozen backbone (including dropout) in evaluation mode.
    if frozen_backbone:
        model.eval()
        model.head.train()
    else:
        model.train()
    total_loss = 0.0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad(set_to_none=True)
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * target.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def test(model, loader):
    model.eval()
    total_loss, correct = 0.0, 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        total_loss += F.cross_entropy(output, target, reduction="sum").item()
        correct += (output.argmax(dim=1) == target).sum().item()
    accuracy = 100.0 * correct / len(loader.dataset)
    print(f"Loss: {total_loss / len(loader.dataset):.4f}, accuracy: {accuracy:.2f}%")
    return accuracy


### Data splits

Each seed selects two classes and disjoint training and validation subsets. Labels are mapped to 0 and 1. Use the same seeds to compare methods on the same problem instances.


In [ ]:
class TwoClassSubset(Dataset):
    def __init__(self, dataset, indices, classes):
        self.dataset = dataset
        self.indices = np.asarray(indices)
        self.label_map = {int(label): i for i, label in enumerate(classes)}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.dataset[int(self.indices[index])]
        return image, self.label_map[int(label)]

def split_indices(targets, seed):
    rng = np.random.RandomState(seed)
    classes = rng.permutation(10)[:2]
    targets = np.asarray(targets)
    train_ids, val_ids = [], []
    for label in classes:
        ids = rng.permutation(np.flatnonzero(targets == label))
        train_ids.extend(ids[:50])
        val_ids.extend(ids[50:400])
    return classes, np.array(train_ids), np.array(val_ids)

def make_loaders(train_data, val_data, seed):
    classes, train_ids, val_ids = split_indices(train_data.targets, seed)
    assert not set(train_ids).intersection(val_ids)
    train_subset = TwoClassSubset(train_data, train_ids, classes)
    val_subset = TwoClassSubset(val_data, val_ids, classes)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE,
                              shuffle=True, generator=generator)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)
    print(f"Classes: {classes}; training: {len(train_subset)}; validation: {len(val_subset)}")
    return train_loader, val_loader


### Small ViT trained from scratch


In [ ]:
def make_scratch_vit():
    # 32 x 32 images, split into 64 patches of size 4 x 4.
    # All parameters, including positional embeddings, start from scratch.
    return VisionTransformer(
        img_size=32, patch_size=4, in_chans=3, num_classes=2,
        embed_dim=128, depth=3, num_heads=4, mlp_ratio=2.0,
        qkv_bias=True, proj_drop_rate=0.1, pos_drop_rate=0.1,
    )


Try one instance while developing your method, then evaluate on multiple instances. The standard deviation below measures variation across the sampled problems, not uncertainty in the mean.


In [ ]:
normalize = transforms.Normalize((0.4914, 0.4822, 0.4465),
                                 (0.247, 0.243, 0.261))
transform_train = transforms.Compose([transforms.ToTensor(), normalize])
transform_val = transforms.Compose([transforms.ToTensor(), normalize])
train_data = datasets.CIFAR10(root="./data", train=True,
                             transform=transform_train, download=True)
val_data = datasets.CIFAR10(root="./data", train=True,
                           transform=transform_val, download=True)

scratch_accs = []
for seed in SEEDS:
    set_seed(seed)
    train_loader, val_loader = make_loaders(train_data, val_data, seed)
    model = make_scratch_vit().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    for epoch in range(SCRATCH_EPOCHS):
        loss = train(model, train_loader, optimizer)
        if epoch % 10 == 0:
            print(f"Seed {seed}, epoch {epoch}: training loss {loss:.4f}")
    scratch_accs.append(test(model, val_loader))
print(f"Scratch ViT: {np.mean(scratch_accs):.2f} +/- {np.std(scratch_accs):.2f}%")


## Challenge 2

Use the same limited training data, but now external pretraining is allowed. Do not use extra CIFAR-10 training images or models pretrained on CIFAR-10.

This example uses a ViT-Tiny pretrained on ImageNet-21k and fine-tuned on ImageNet-1k. It freezes the backbone and trains a new two-class classification head. Its image preprocessing comes from the checkpoint configuration. You may explore different fine-tuning strategies.

[Checkpoint and preprocessing](https://huggingface.co/timm/vit_tiny_patch16_224.augreg_in21k_ft_in1k)


In [ ]:
MODEL_NAME = "vit_tiny_patch16_224.augreg_in21k_ft_in1k"

def make_transfer_vit(pretrained=True):
    model = timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=2)
    model.requires_grad_(False)
    model.head.requires_grad_(True)
    return model

# Retrieve the checkpoint's expected resize and normalization settings.
# pretrained=False here avoids downloading weights just to inspect preprocessing.
config_model = timm.create_model(MODEL_NAME, pretrained=False)
data_config = timm.data.resolve_model_data_config(config_model)
transfer_transform = timm.data.create_transform(**data_config, is_training=False)
del config_model


In [ ]:
train_data = datasets.CIFAR10(root="./data", train=True,
                             transform=transfer_transform, download=True)
val_data = datasets.CIFAR10(root="./data", train=True,
                           transform=transfer_transform, download=True)

transfer_accs = []
for seed in SEEDS:
    set_seed(seed)
    train_loader, val_loader = make_loaders(train_data, val_data, seed)
    # Reinitialize the head and reload the pretrained backbone for each instance.
    model = make_transfer_vit().to(device)
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=0.01)
    for epoch in range(TRANSFER_EPOCHS):
        loss = train(model, train_loader, optimizer, frozen_backbone=True)
        print(f"Seed {seed}, epoch {epoch}: training loss {loss:.4f}")
    transfer_accs.append(test(model, val_loader))
print(f"Pretrained ViT + head: {np.mean(transfer_accs):.2f} +/- {np.std(transfer_accs):.2f}%")
